In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv(r"C:\Users\vishwa\Downloads\archive (12)\AB_NYC_2019.csv")

In [3]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [4]:
df["name"].fillna("Unknown", inplace=True)
df["host_name"].fillna("Unknown", inplace=True)
df["last_review"].fillna(0, inplace=True)
df["reviews_per_month"].fillna(0, inplace=True)

C:\Users\vishwa\AppData\Local\Temp\ipykernel_12844\2111749131.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["name"].fillna("Unknown", inplace=True)
C:\Users\vishwa\AppData\Local\Temp\ipykernel_12844\2111749131.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48895 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48895 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

In [5]:
X = df[['latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 
        'calculated_host_listings_count', 'availability_365','neighbourhood_group', 'room_type','neighbourhood']]
y = df[['price']]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

# TRAIN DATA PREPROCESSING

In [7]:
num_df_train = X_train[['latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 
                        'reviews_per_month', 'calculated_host_listings_count', 'availability_365']]
cat_df_train = X_train[['neighbourhood_group','neighbourhood']]
cat_dfo_train = X_train[['room_type']]

In [8]:
OHE = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
OHE.fit(cat_df_train)
cat_train_trans = OHE.transform(cat_df_train)

In [9]:
OE = OrdinalEncoder()
OE.fit(cat_dfo_train)
cato_df_trans_train = OE.transform(cat_dfo_train)

In [10]:
STD = StandardScaler()
STD.fit(num_df_train)
num_df_trans_train = STD.transform(num_df_train)

In [11]:
STD_y = StandardScaler()
y_train_trans = STD_y.fit_transform(y_train)

In [12]:
cat_train_trans = pd.DataFrame(cat_train_trans)
cato_df_trans_train = pd.DataFrame(cato_df_trans_train)
num_df_trans_train = pd.DataFrame(num_df_trans_train)
y_train_trans = pd.DataFrame(y_train_trans)

cat_train_trans.reset_index(inplace=True)
cato_df_trans_train.reset_index(inplace=True)
num_df_trans_train.reset_index(inplace=True)
y_train_trans.reset_index(inplace=True)

X_train_trans = pd.concat([cat_train_trans, cato_df_trans_train, num_df_trans_train], axis=1)
X_train_trans.pop('index')
y_train_trans.pop('index')

0            0
1            1
2            2
3            3
4            4
         ...  
39111    39111
39112    39112
39113    39113
39114    39114
39115    39115
Name: index, Length: 39116, dtype: int64

# TEST DATA PREPROCESSING

In [13]:
num_df_test = X_test[['latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 
                      'reviews_per_month', 'calculated_host_listings_count', 'availability_365']]
cat_df_test = X_test[['neighbourhood_group','neighbourhood']]
cat_dfo_test = X_test[['room_type']]

In [14]:
cat_test_trans = OHE.transform(cat_df_test)
cato_df_trans_test = OE.transform(cat_dfo_test)
num_df_trans_test = STD.transform(num_df_test)
y_test_trans = STD_y.transform(y_test)

In [15]:
cat_test_trans = pd.DataFrame(cat_test_trans)
cato_df_trans_test = pd.DataFrame(cato_df_trans_test)
num_df_trans_test = pd.DataFrame(num_df_trans_test)
y_test_trans = pd.DataFrame(y_test_trans)

cat_test_trans.reset_index(inplace=True)
cato_df_trans_test.reset_index(inplace=True)
num_df_trans_test.reset_index(inplace=True)
y_test_trans.reset_index(inplace=True)

X_test_trans = pd.concat([cat_test_trans, cato_df_trans_test, num_df_trans_test], axis=1)
X_test_trans.pop('index')
y_test_trans.pop('index')

0          0
1          1
2          2
3          3
4          4
        ... 
9774    9774
9775    9775
9776    9776
9777    9777
9778    9778
Name: index, Length: 9779, dtype: int64

# TRAIN KNN MODEL

In [16]:
model = KNeighborsRegressor(n_neighbors=5)
model.fit(X_train_trans, y_train_trans.values.flatten())

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


# MAKE PREDICTIONS

In [17]:
y_pred = model.predict(X_test_trans)

# EVALUATE

In [20]:
from sklearn.metrics import accuracy_score
import numpy as np

acc = accuracy_score(y_test_trans, y_pred)
print(acc)

ValueError: continuous is not supported

In [21]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("R2 Score:", r2_score(y_test_trans, y_pred))
print("MAE:", mean_absolute_error(y_test_trans, y_pred))
print("MSE:", mean_squared_error(y_test_trans, y_pred))

R2 Score: 0.05633685277073386
MAE: 0.2698432174397667
MSE: 0.6840669390499844


In [23]:
import joblib
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
joblib.dump({
    'model': model,
    'num_encod': STD,
    'num_encod_y': STD_y,
    'cat_encod': OHE,
    'cato_encod': OE
}, 'airbnb_knn_model.pkl')

['airbnb_knn_model.pkl']